# 🧠 Tutorial 1 — Python Basics & EEG Data Loading
### · Biomedical Signals & Applications · EEG Python Workshop

---

**Goal of this tutorial:** Get comfortable with Python and load real EEG data for the first time.

**By the end you will be able to:**
- Use Python variables, lists, dictionaries, loops, and functions
- Create and manipulate NumPy arrays (the backbone of all scientific Python)
- Load a real EEG recording into Python using MNE
- Understand what is stored in a Raw EEG object
- Plot a raw EEG trace, the electrode montage, and the power spectrum

**Time:** approximately 60 minutes

---
> 💡 **Before running any cell:** make sure you have installed the required libraries:
> ```
> pip install mne numpy matplotlib scipy
> ```
> Run cells one by one from top to bottom using **Shift + Enter**.

---
## Section 0 — Python Crash Course

If you have never used Python before, this section gives you the minimum you need for EEG analysis. Work through each sub-section — it only takes about 10 minutes.

### 0.1 Variables and Basic Types

Python stores values in **variables**. The type of a variable (number, text, true/false) is determined automatically.

In [ ]:
name    = "EEG Workshop"   # string (text)
year    = 2025             # integer (whole number)
fs      = 256.0            # float (decimal number — sampling frequency in Hz)
is_demo = True             # boolean (True or False)

print(name, year, fs, is_demo)

### 0.2 Lists & Indexing

A **list** is an ordered collection of items. Python uses **0-based indexing** — the first element is at position 0, not 1.

In [ ]:
channels = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2"]

print("First channel :", channels[0])   # position 0 = first
print("Last channel  :", channels[-1])  # -1 = last element
print("First three   :", channels[:3])  # slice: from start up to (not including) index 3

### 0.3 Dictionaries

A **dictionary** stores key–value pairs. Perfect for metadata like subject information.

In [ ]:
subject_info = {
    "id"   : "S01",
    "age"  : 22,
    "sex"  : "M",
    "task" : "motor imagery",
}
print("Task:", subject_info["task"])
print("Age :", subject_info["age"])

### 0.4 Loops

Loops let you repeat an action for every item in a list, or a fixed number of times.

In [ ]:
# Loop over a list
for ch in channels[:3]:
    print("Channel:", ch)

# Loop over a range of numbers (0, 1, 2, 3, 4)
for i in range(5):
    print(i, end=" ")
print()  # newline

### 0.5 Functions

A **function** packages a reusable block of code. You define it once with `def`, then call it as many times as you need.

In [ ]:
def band_name(low, high):
    """Return the EEG frequency band name for a given range."""
    if high <= 4:
        return "Delta"
    elif high <= 8:
        return "Theta"
    elif high <= 13:
        return "Alpha"
    elif high <= 30:
        return "Beta"
    else:
        return "Gamma"

print(band_name(8, 13))   # → Alpha
print(band_name(1,  4))   # → Delta
print(band_name(13, 30))  # → Beta

### 0.6 NumPy — The Backbone of Scientific Python

**NumPy** is the library that makes Python fast for numerical work. Instead of looping over thousands of numbers one by one, NumPy operates on whole arrays at once.

> 🟢 **Analogy:** Think of a NumPy array as a spreadsheet column. Instead of writing a formula for each cell, you write one formula and it applies to all cells simultaneously.

In [ ]:
import numpy as np

# Create a time axis: 1 second sampled at 256 Hz = 256 points
t = np.linspace(0, 1, 256)      # from 0 to 1 second, 256 equally spaced points
f = 10.0                          # 10 Hz = alpha band frequency

# Generate a pure sine wave (what a single EEG oscillation looks like)
signal = np.sin(2 * np.pi * f * t)

print("Array shape :", signal.shape)             # (256,) = 256 numbers
print("Mean        :", round(np.mean(signal), 4)) # should be ≈ 0
print("Std         :", round(np.std(signal), 4))  # should be ≈ 0.707
print("Min / Max   :", round(signal.min(), 4), "/", round(signal.max(), 4))

### 0.7 Quick Matplotlib Plot

Let's visualise the sine wave we just created. This is how every EEG waveform will be displayed.

In [ ]:
import matplotlib.pyplot as plt

signal_uV = signal * 50.0   # scale to ±50 µV (realistic EEG amplitude)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, signal_uV, color="steelblue", linewidth=1.5)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (µV)")
ax.set_title("Synthetic 10 Hz Sine Wave — This is what the Alpha rhythm looks like")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Section 1 — Introduction to MNE-Python

**MNE-Python** is the gold-standard open-source library for EEG and MEG analysis. It handles:
- Loading EEG from virtually any file format (EDF, BDF, FIF, BrainVision, ...)
- Preprocessing (filtering, ICA, epoching)
- Visualisation (raw traces, topographic maps, time-frequency plots)
- Analysis (ERPs, connectivity, decoding)

We will use MNE for everything from this point forward.

In [ ]:
import mne

print("MNE version:", mne.__version__)
mne.set_log_level("WARNING")   # turn off verbose output during the tutorial

---
## Section 2 — Loading Real EEG Data

We will use the **PhysioNet EEG Motor Movement/Imagery Dataset** — a publicly available dataset where subjects imagined moving their left or right hand. MNE can download it automatically.

- **Subject:** 1
- **Runs:** 6 and 10 (these contain motor imagery trials: left hand = T1, right hand = T2)
- **Format:** EDF (European Data Format — the standard for clinical EEG)
- **Channels:** 64 electrodes
- **Sampling rate:** 160 Hz

> 🟢 **Analogy:** Loading the raw EEG is like opening a very long spreadsheet — 64 columns (channels) and ~20,000 rows (time samples). MNE reads the file and packages everything into a single convenient object called `raw`.

In [ ]:
from mne.datasets import eegbci
from mne.io import concatenate_raws, read_raw_edf

subject = 1
runs    = [6, 10]   # motor imagery: left hand (T1) and right hand (T2)

# Download data if not already cached, then load
raw_files = eegbci.load_data(subject, runs, verbose=False)
raw = concatenate_raws([read_raw_edf(f, preload=True, verbose=False)
                        for f in raw_files])

# Standardise channel names (PhysioNet uses non-standard capitalisations)
eegbci.standardize(raw)

# Attach electrode positions (needed for topographic maps later)
montage = mne.channels.make_standard_montage("standard_1005")
raw.set_montage(montage, verbose=False)

print("--- Raw object ---")
print(raw)

> 👁 **What to check:**
> - `n_channels = 64` — we have 64 EEG electrodes
> - `sfreq = 160.0 Hz` — 160 samples per second per channel
> - Duration should be about 2 minutes (two runs of ~62 seconds each)

---
## Section 3 — Exploring Raw EEG Data

### 3.1 Accessing the Data as a NumPy Array

The raw EEG data is stored internally as a 2D NumPy array:
- **Rows** = channels (64 electrodes)
- **Columns** = time samples (~19,840 samples at 160 Hz × ~124 seconds)

In [ ]:
data, times = raw.get_data(return_times=True)

print("Data shape (channels × samples) :", data.shape)
print("Time axis shape                  :", times.shape)
print(f"Time from {times[0]:.2f} s to {times[-1]:.2f} s")

# Look at one channel
ch0 = data[0]
print(f"\nChannel '{raw.ch_names[0]}':")
print(f"  Mean : {ch0.mean()*1e6:.2f} µV")   # convert V → µV by ×1e6
print(f"  Std  : {ch0.std()*1e6:.2f} µV")

### 3.2 Plot a Few Seconds of Raw EEG

This shows the raw voltage traces for 10 channels over 5 seconds. Notice:
- The signals are messy — lots of noise and artefacts on top of the brain signal
- Large spikes at frontal channels (Fp1, Fp2) are eye blinks
- The amplitude is in µV — very small numbers!

In [ ]:
fig = raw.plot(
    duration=5,
    n_channels=15,
    scalings="auto",
    title="Raw EEG — Motor Imagery (first 5 seconds)",
    show=True,
    block=False,
)
plt.show()

### 3.3 Plot the Electrode Montage

This shows the positions of all 64 electrodes on a 2D projection of the scalp (viewed from above). Key electrodes for motor imagery:
- **C3** (left): over the left motor cortex → active during RIGHT hand imagery
- **Cz** (centre): over the central motor cortex
- **C4** (right): over the right motor cortex → active during LEFT hand imagery

In [ ]:
fig = raw.plot_sensors(show_names=True, show=True)
plt.show()

### 3.4 Power Spectral Density — First Look at Frequencies

The **Power Spectral Density (PSD)** shows how much signal energy is present at each frequency. This is our first look at the frequency content of the EEG.

> 🟢 **Analogy:** The PSD is like a music equaliser showing the volume at each frequency band. We can see whether there is more bass (low frequencies) or treble (high frequencies) in the signal.

**What to look for:**
- The signal decreases as frequency increases (this is normal — called 1/f or "pink noise")
- There may be a bump at 8–12 Hz (the **alpha peak** — the idle rhythm of the brain)
- There will be a sharp spike at **60 Hz** (US power-line interference — we will remove this in Tutorial 2)

In [ ]:
psd = raw.compute_psd(fmax=80)
fig = psd.plot(show=True)
plt.show()

---
## Section 4 — The MNE Info Object

`raw.info` is a dictionary-like container with all the metadata about the recording. You will use this constantly in MNE.

In [ ]:
print("Sampling frequency  :", raw.info["sfreq"], "Hz")
print("Number of channels  :", raw.info["nchan"])
print("First 5 channels    :", raw.info["ch_names"][:5])
print("Marked bad channels :", raw.info["bads"])   # empty for now
print("\nFull info object:")
print(raw.info)

---
## Section 5 — Events and Annotations

During recording, **event markers** (also called annotations) were saved at the exact moment each experimental event occurred:
- **T0** = rest period begins
- **T1** = left hand imagery cue appeared
- **T2** = right hand imagery cue appeared

`mne.events_from_annotations()` extracts these time stamps as an array with 3 columns:
1. Sample number (when the event happened)
2. Previous event ID (usually 0)
3. Event ID (1=T0, 2=T1, 3=T2)

In [ ]:
events, event_id = mne.events_from_annotations(raw, verbose=False)

print("Event ID dictionary:", event_id)
print("Events array shape :", events.shape)   # (n_events, 3)
print("\nFirst 10 events (sample, prev_id, event_id):")
print(events[:10])

In [ ]:
# Visualise event timing on a timeline
fig = mne.viz.plot_events(
    events, event_id=event_id, sfreq=raw.info["sfreq"],
    first_samp=raw.first_samp
)
plt.title("Event markers in the recording")
plt.show()

> 👁 **What to check:**
> - T1 (left hand) and T2 (right hand) should alternate regularly
> - Each trial lasts about 4 seconds (you can check the sample numbers)
> - This structure will be used in Tutorial 2 to cut the recording into epochs

---
## Section 6 — Manual Plot of One Channel

Let's plot a short segment of channel C3 manually using matplotlib, to understand exactly what we are working with.

**C3** sits over the LEFT motor cortex — it is one of the two most important channels for motor imagery classification.

In [ ]:
idx_c3 = raw.ch_names.index("C3")
data_c3 = data[idx_c3, :800] * 1e6   # first 5 seconds, converted to µV
time_c3 = times[:800]

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(time_c3, data_c3, color="steelblue", lw=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (µV)")
ax.set_title("Channel C3 — Raw EEG (first 5 seconds, before preprocessing)")
ax.axhline(0, color="k", lw=0.5, linestyle="--")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print(f"C3 amplitude range: {data_c3.min():.1f} to {data_c3.max():.1f} µV")

---
## ✅ Tutorial 1 — Summary

You have completed Tutorial 1. Here is what you learned:

| Concept | Python / MNE command |
|---|---|
| NumPy array creation | `np.linspace()`, `np.sin()` |
| Array statistics | `np.mean()`, `np.std()` |
| Basic plotting | `plt.plot()`, `plt.show()` |
| Load EEG from file | `read_raw_edf()`, `concatenate_raws()` |
| Attach electrode positions | `raw.set_montage()` |
| Access raw data | `raw.get_data()` |
| Inspect metadata | `raw.info` |
| Extract event markers | `mne.events_from_annotations()` |
| Quick spectrum plot | `raw.compute_psd().plot()` |

---

### ➜ Next: Tutorial 2 — EEG Preprocessing

In Tutorial 2 we will clean this raw data:
1. Re-reference to average
2. Filter (remove power-line noise and slow drifts)
3. Detect and interpolate bad channels
4. Remove eye-blink and muscle artefacts with ICA
5. Cut into epochs around the motor imagery cues
6. Apply baseline correction and reject noisy trials